# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
# !pip install --upgrade transformers datasets accelerate deepspeed --no-cache
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

### Data Preparation

In [3]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [4]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

In [5]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [6]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

In [7]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [8]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [9]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [10]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
device

'cuda:0'

In [11]:
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16, shuffle=False, collate_fn=transformers.default_data_collator, num_workers=8
)
model.to(device);

In [12]:
from tqdm.notebook import tqdm

def get_accuracy(model, dataloader):
    model.eval()
    device = model.device

    accuracy = 0
    n = 0
    with torch.no_grad():
        for batch in tqdm(dataloader):
            mask = batch["attention_mask"].to(device)
    
            if "token_type_ids" in batch:
                preds = model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=mask,
                    token_type_ids=batch["token_type_ids"].to(device)
                ).logits.argmax(dim=-1)
            else:
                preds = model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=mask
                ).logits.argmax(dim=-1)
    
            target = batch["labels"].to(device)
            accuracy += (preds == target).sum()
            n += preds.shape[0]
    return accuracy / n

In [13]:
accuracy = get_accuracy(model, val_loader)

  0%|          | 0/2527 [00:00<?, ?it/s]

In [14]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

I tried option A, got $0.81$ accuracy after 6 hours of training and decided to switch to option B.

#### Option A attempt (not graded)

In [15]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(p):
    y_pred, y_true = p
    y_pred = np.argmax(y_pred, axis=1)

    return {
        "accuracy": (y_pred == y_true).mean(),
        "f1": f1_score(y_true, y_pred, average="binary")
    }

In [16]:
from transformers import TrainingArguments, Trainer

model_name = "microsoft/deberta-v3-base"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

batch_size = 16
num_epochs = 4
total_steps = num_epochs * (len(qqp_preprocessed["train"]) // batch_size)

training_args = TrainingArguments(
    output_dir="./deberta-dup",
    eval_strategy="steps",
    eval_steps=5000,
    logging_steps=5000,
    logging_dir="./logs",
    report_to="none",

    learning_rate=5e-5,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    lr_scheduler_type=transformers.SchedulerType.COSINE,
    warmup_steps=int(0.1 * total_steps),
    weight_decay=0.1
)

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     data_collator=transformers.default_data_collator,
#     train_dataset=qqp_preprocessed["train"],
#     eval_dataset=qqp_preprocessed["validation"],
#     compute_metrics=compute_metrics
# )

/home/vagiz/Desktop/desktop_vagiz/yandex/nlp_course/.venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
# trainer.train()

In [18]:
# trainer.save_model("./deberta-finetuned-qqp-my-1")

#### Option B

In [19]:
model_names = (
    ("gchhablani/bert-base-cased-finetuned-qqp", "BERT"),
    ("howey/electra-small-qqp", "ELECTRA"),
    ("tanganke/gpt2_qqp", "GPT 2"),
    ("textattack/albert-base-v2-QQP", "ALBERT")
)

In [20]:
import os
import tempfile
import time

BATCH_SIZE = 16
NUM_WORKERS = 8

def get_qqp(model_name):
    qqp = datasets.load_dataset("SetFit/qqp")
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    
    def preprocess_function(examples):
        result = tokenizer(
            examples["text1"],
            examples["text2"],
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True,
        )
    
        result["label"] = examples["label"]
    
        return result

    return qqp.map(preprocess_function, batched=True)

def measure_model(model_name):
    qqp_preprocessed = get_qqp(model_name)

    dataloader = torch.utils.data.DataLoader(
        qqp_preprocessed["validation"],
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=transformers.default_data_collator,
        num_workers=NUM_WORKERS
    )

    model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Number of parameters
    num_params = sum(p.numel() for p in model.parameters())

    # Weights size (32 bits per float -> 32 / 8 = 4 bytes per float)
    size_mb = num_params * 4 / (1024 * 1024)

    # Total size (including extra files)
    with tempfile.TemporaryDirectory() as tmpdirname:
        model.save_pretrained(tmpdirname)
        size_bytes = sum(os.path.getsize(os.path.join(root, file))
                         for root, _, files in os.walk(tmpdirname)
                         for file in files)
    total_size_mb = size_bytes / (1024 * 1024)

    # Accuracy and speed
    start = time.time()
    acc = get_accuracy(model, dataloader)
    time_taken = time.time() - start

    total_samples = len(dataloader.dataset)
    speed = total_samples / time_taken if time_taken > 0 else 0

    return acc, speed, total_size_mb, size_mb, num_params

In [21]:
# all_metrics = {}

# for model_name, model_desc in model_names:
#     acc, speed, total_size_mb, size_mb, num_params = measure_model(model_name)
#     all_metrics[model_desc] = {
#         "Accuracy": f"{100 * acc:.1f}%",
#         "Speed": f"{speed:.1f}",
#         "Total Size": f"{int(total_size_mb)} MB",
#         "Weights Size": f"{int(size_mb)} MB",
#         "# of Parameters": f"{num_params / 1_000_000:.1f}M"
#     }

##### Hardware

| GPU |
|---|
| NVIDIA GeForce RTX 4070 Mobile 8GB |

| CPU |
|---|
| AMD Ryzen 9 8945H w/ Radeon 780M Graphics |

| OS | CUDA |  |  |  |
|---|---|---|---|---|
| Ubuntu 24.04.3 LTS | cuda_12.0.r12.0 |  |  |  |





##### Inference Parameters

| batch_size | num_workers |
|---|---|
| 16 | 8 |

In [22]:
# import pandas as pd

# pd.DataFrame(all_metrics).to_csv("all_metrics.csv")

In [23]:
import pandas as pd

pd.read_csv("all_metrics.csv", index_col="Unnamed: 0")

,BERT,ELECTRA,GPT 2,ALBERT
Accuracy,90.8%,89.5%,89.6%,90.6%
Speed,263.4,1408.9,220.7,207.0
Total Size,413 MB,51 MB,474 MB,44 MB
Weights Size,413 MB,51 MB,474 MB,44 MB
# of Parameters,108.3M,13.5M,124.4M,11.7M


P.S. Total size - size of folder with saved model, weights size - parameters considered only.

##### Report

Core findings:

* BERT is the best model among chosen, with ALBERT on the second place ($0.2\%$ lower in accuracy) and ELECTRA/GPT-2 close with $\approx 89.5\%$ performance.
* The fastest model is ELECTRA, which is $\approx \times 5$ faster and $\approx \times 8$ times smaller, compared to BERT.
* ALBERT model is the smallest and the slowest model, which is counter intuitive at first. However, it is expected if knowing what ALBERT is: it is BERT with deeper structure but shared weights accross layers, making it much more efficient in terms of size but deeper to maintain solid performance, therefore, more expensive from the compute perspective.
* GPT-2 is $15\%$ bigger model than BERT, with a tradeoff of being $16\%$ slower. However, its performance is less than BERT's by $1.2\%$ in accuracy.

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

Since ELECTRA works almost on the same level as BERT but much faster, I will use it for the task.

I will use only `text1` from each pair in QQP.

In [24]:
del model
torch.cuda.empty_cache()

In [25]:
import torch.nn.functional as F


def collate_fn(batch):
    model_inputs = transformers.default_data_collator([
        {k: v for k, v in item.items() if k in ["input_ids", "attention_mask", "token_type_ids", "labels"]}
        for item in batch
    ])
    model_inputs["text2"] = [item["text2"] for item in batch]
    return model_inputs


def get_duplicates(question: str, model_name: str = "howey/electra-small-qqp", k: int = 5):
    device = "cuda:0"

    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

    qqp = datasets.load_dataset("SetFit/qqp")

    def preprocess_duplicates(examples):
        question_batch = [question] * len(examples["text2"])

        result = tokenizer(
            question_batch,
            examples["text2"],
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True,
        )
        result["text2"] = examples["text2"]
        return result

    database = qqp['train'].map(preprocess_duplicates, batched=True, num_proc=NUM_WORKERS)
    database.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "text2"])
    dataloader = torch.utils.data.DataLoader(
        database,
        batch_size=64,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS
    )

    model.eval()
    model.to(device)
    probs = []
    texts = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Ranking duplicates..."):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            cur_logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            ).logits
            cur_probs = F.softmax(cur_logits, dim=1)[:, 1].cpu().numpy()  # probability of duplicate
            probs.append(cur_probs)
            texts.extend(batch["text2"])

    probs = np.concatenate(probs)
    texts = np.array(texts)

    top_k_indices = probs.argsort()[::-1][:k]

    top_k = []
    for idx in top_k_indices:
        text = str(texts[idx])
        prob = float(probs[idx])
        top_k.append((text, prob))

    return top_k

In [26]:
def find_similar_questions(questions, fn):
    pairs = {}
    for question in questions:
        start = time.time()
        top_k = fn(question, k=5)
        total_time = time.time() - start
        pairs[question] = top_k, total_time
    return pairs

def show_pairs(pairs):
    for question, (top_k, total_time) in pairs.items():
        print(f"Q: {question}")
        print(f"Time: {int(total_time) // 60}m {int(total_time) % 60}s")
        for (text, prob) in top_k:
            print(f"    prob: {prob:.2f}, text: {text}")
        print()

In [27]:
questions = (
    "What is the best feeling in your life?",
    "How can you learn to cook?",
    "What country is best for life?",
    "Is MIT worth it?",
    "What is the fastest way to raise money?"
)

In [28]:
pairs_naive = find_similar_questions(questions, fn=get_duplicates)
show_pairs(pairs_naive)

Repo card metadata block was not found. Setting CardData to empty.


Ranking duplicates...:   0%|          | 0/5686 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


Ranking duplicates...:   0%|          | 0/5686 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


...

Repo card metadata block was not found. Setting CardData to empty.


Ranking duplicates...:   0%|          | 0/5686 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


Ranking duplicates...:   0%|          | 0/5686 [00:00<?, ?it/s]

Q: What is the best feeling in your life?
Time: 3m 58s
    prob: 0.98, text: What was your best feeling in the world?
    prob: 0.98, text: What was your best feeling in the world?
    prob: 0.98, text: What was your best feeling in the world?
    prob: 0.98, text: What was your best feeling in the world?
    prob: 0.98, text: What is If there was one thing you would like to change about Quora what would it be? That one thing where Quora need to improve?

Q: How can you learn to cook?
Time: 4m 1s
    prob: 0.98, text: Why did Indian Government introduced ₹2000 note instead of the new ₹1000 note? Meanwhile, they introduced the new ₹500 note for old ₹500 note.
    prob: 0.98, text: Why did Indian Government introduced ₹2000 note instead of the new ₹1000 note? Meanwhile, they introduced the new ₹500 note for old ₹500 note.
    prob: 0.97, text: Why did the government print Rs 2000 notes? Why they didn’t print new 1000 notes?
    prob: 0.97, text: Why did the government print Rs 2000 notes

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

#### Optimization

**Idea**: filter out some samples from train dataset, then apply model on filtered part. How to filter out samples? For example, leave only samples that have at leat 1 word of intersection with the input question.

1. We use `nltk.tokenize.word_tokenize` to get tokens fast and compute intersection with input question.
2. In order to handle corner cases, we restrict the minimum number of samples left after restriction. If there is too few left, we recursively try with softer constraints.
3. Run previous solution on filtered dataset.

In [29]:
from nltk.tokenize import word_tokenize
import nltk
nltk.download("punkt_tab")

def get_duplicates_with_filter(
    question: str,
    model_name: str = "howey/electra-small-qqp",
    k: int = 5,
    min_intersection: int = 4,
    min_database_size: int = 100
):
    device = "cuda:0"

    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

    question_words = set(word_tokenize(question.lower()))
    min_intersection = min(min_intersection, max(1, len(question_words) - 1))
    def has_overlap(example):
        text_words = set(word_tokenize(example["text2"].lower()))
        return len(question_words & text_words) >= min_intersection

    qqp = datasets.load_dataset("SetFit/qqp")
    train_filtered = qqp.filter(has_overlap, num_proc=NUM_WORKERS)['train']
    if len(train_filtered) < min_database_size:
        if min_intersection > 1:
            return get_duplicates_with_filter(question, model_name, k, min_intersection - 1, min_database_size)
        else:
            raise RuntimeError("Question contains unknown language.")

    def preprocess_duplicates(examples):
        question_batch = [question for _ in examples["text1"]]

        result = tokenizer(
            examples["text1"],
            question_batch,
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True,
        )
        return result

    database = train_filtered.map(preprocess_duplicates, batched=True, num_proc=NUM_WORKERS)
    database.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "text2"])
    dataloader = torch.utils.data.DataLoader(
        database,
        batch_size=64,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS
    )

    model.eval()
    model.to(device)
    probs = []
    texts = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Ranking duplicates..."):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            cur_logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            ).logits
            cur_probs = F.softmax(cur_logits, dim=1)[:, 1].cpu().numpy()  # probability of duplicate
            probs.append(cur_probs)
            texts.extend(batch["text2"])

    probs = np.concatenate(probs)
    texts = np.array(texts)

    top_k_indices = probs.argsort()[::-1][:k]

    top_k = []
    for idx in top_k_indices:
        text = str(texts[idx])
        prob = float(probs[idx])
        top_k.append((text, prob))

    return top_k

[nltk_data] Downloading package punkt_tab to /home/vagiz/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [30]:
pairs_with_filter = find_similar_questions(questions, fn=get_duplicates_with_filter)
show_pairs(pairs_with_filter)

Repo card metadata block was not found. Setting CardData to empty.


Ranking duplicates...:   0%|          | 0/1232 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


Ranking duplicates...:   0%|          | 0/165 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


...

Repo card metadata block was not found. Setting CardData to empty.


Ranking duplicates...:   0%|          | 0/9 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


Ranking duplicates...:   0%|          | 0/943 [00:00<?, ?it/s]

Q: What is the best feeling in your life?
Time: 0m 53s
    prob: 0.98, text: What was the best crush experience you've ever had?
    prob: 0.98, text: What is the most embarrassing moment of your life?
    prob: 0.98, text: What are your best experiences of life and why?
    prob: 0.98, text: If energy is not conserved in an expanding universe, is potential energy infinite (the energy that can be created is infinite)?
    prob: 0.98, text: What are some of the best feelings in life?

Q: How can you learn to cook?
Time: 0m 9s
    prob: 0.96, text: How can discontinuing 500 and 1000 rupee will help to control black money?
    prob: 0.90, text: How do I learn to cook?
    prob: 0.85, text: What are the best ways to learn to cook?
    prob: 0.85, text: How do I learn to cook?
    prob: 0.85, text: How did you learn to cook?

Q: What country is best for life?
Time: 0m 17s
    prob: 0.98, text: What is the best place to travel?
    prob: 0.98, text: Which is the best country in the world to 

I compare three approaches in the final bonus task.

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

**Idea**: download pretrained embeddings (I stick to GloVe), then precompute mean embedding of each sentence in train. Use dot-product to find most similar sample for input sentence.

In [31]:
def get_mean_emb(text, glove):
    words = text.lower().split()
    vectors = [glove[w] for w in words if w in glove]
    if not vectors:
        return np.zeros(glove.vector_size)
    return np.mean(vectors, axis=0)

In [32]:
from gensim.models.keyedvectors import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

# glove2word2vec('glove.6B.300d.txt', 'glove.6B.300d.word2vec.txt')
glove = KeyedVectors.load_word2vec_format('glove.6B.300d.word2vec.txt', binary=False)

In [33]:
qqp = datasets.load_dataset("SetFit/qqp")
train_dataset = qqp["train"]

precomputed = []
for example in tqdm(train_dataset, desc="Precomputing embeddings"):
    emb = get_mean_emb(example["text2"], glove)
    precomputed.append(emb)
precomputed = np.array(precomputed)

norm = np.linalg.norm(precomputed, axis=1, keepdims=True)
norm[norm == 0] = 1
precomputed_normed = precomputed / norm

Repo card metadata block was not found. Setting CardData to empty.


Precomputing embeddings:   0%|          | 0/363846 [00:00<?, ?it/s]

In [34]:
def get_duplicates_with_glove(question: str, k: int = 5):
    query_emb = get_mean_emb(question, glove)
    query_normed = query_emb / np.linalg.norm(query_emb)

    scores = np.dot(precomputed_normed, query_normed)

    indexes = np.argsort(scores)[-k:][::-1]

    top_k = []
    for i in indexes:
        text = train_dataset[i]["text2"]
        score = scores[i]
        top_k.append((text, score))

    return top_k

In [35]:
pairs_with_glove = find_similar_questions(questions, fn=get_duplicates_with_glove)
show_pairs(pairs_with_glove)

Q: What is the best feeling in your life?
Time: 0m 0s
    prob: 0.98, text: What was your best feeling in the world?
    prob: 0.98, text: What was your best feeling in the world?
    prob: 0.98, text: What was your best feeling in the world?
    prob: 0.98, text: What was your best feeling in the world?
    prob: 0.97, text: What is the best thing you have in your relationship?

Q: How can you learn to cook?
Time: 0m 0s
    prob: 0.99, text: How can I learn to draw?
    prob: 0.99, text: How can I learn to draw?
    prob: 0.99, text: How can I learn to hack?
    prob: 0.99, text: How can I learn to hack?
    prob: 0.99, text: How can I learn to draw?

Q: What country is best for life?
Time: 0m 0s
    prob: 0.99, text: What is the best country for migration?
    prob: 0.99, text: What is the best country for immigration?
    prob: 0.99, text: What is the best country for migration?
    prob: 0.97, text: Which country does what best for jewelry?
    prob: 0.97, text: What is the best so

#### Duplicates Finding Comparison

Summary (quality is discussed further):

| Method                | Speed                          | GPU                     | CPU                                      | RAM        |
|-----------------------|--------------------------------|-------------------------|------------------------------------------|------------|
| ELECTRA (naive)       | Slow (4m per question)         | Required, high load     | Required, batching & mapping             | Low Usage  |
| ELECTRA (with filter) | Moderate (10-60s per question) | Required, moderate load | Required, batching & mapping & filtering | Low Usage  |
| Glove (dot product)   | Fast (1m load & precompute,    | Not required            | Required, dot product                    | Moderate Usage |

Detailed comparison:

* Naive method uses GPU a lot and conducts brute force search. It is the slowest approach and has the worst quality. The problem is that the input question could be out of QQP train distribution, resulting in weird model predictions.
* Filtering allows to narrow down the scope and leave somewhat relevant part of training set for the model. This approach is several times faster than naive one, but has a drawback: filtering based on key words does not always work. The last question about money is a great example: "is", "to", and "money" appeared in different context (Indian finance), that has completely different semantic meaning. Thus, overall quality is about the naive model, only speed is better.
* Final method with embeddings is a great choice since mean embeddings are good for short sequences. We see that all the questions are answered correctly except for the "How can I learn to cook?" question. This example shows a potential issue of this approach: questions with many similar words and a single different word will have high dot product, resulting in out-of-meaning answers. However, we also see that "Is MIT worth it?" has the best answer "What is mit like?", which is the best answer from the dataset (I guess).
* Finally, the embeddings method requires only RAM, no GPU, and uses the vectorized dot product only. RAM/CPU is much cheaper than GPU inference, and it is easier to scale up (for instance, if high RPS is needed). Thus, I believe that this is the best option among considered.

Potential improvement:
* Probably, computing similarity with unnormalized embeddings would be a better option. Initially, in Word2Vec, raw (unscaled) dot-products instead of cosine similarity where chosen so that model has an additional degree of freedom - embedding norm. Thus, raw dot-products may have different performance, but I guess it requires more detailed investigation that looking at 5 random questions.